In [96]:
!pip install lxml
!pip install plotly


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Obtaining dependency information for plotly from https://files.pythonhosted.org/packages/e5/7d/905a3a3d51087515719058c94cfbda2ff0fc14417c20d557ae3e82d8b250/plotly-7.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=1.15.1 from https://files.pythonhosted.org/packages/40/b5/1b84b2c784db76d69442334bc8b8748c840f13ca53be086f4f250ad4a0bc/narwhals-2.26.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 22.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.0/474.0 kB 11.1 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


# Part 1

In [127]:
# web scraping
import requests as rq
import bs4
import pandas as pd
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

# read in the webpage
page = rq.get(url, headers={"User-Agent": "Chrome"})

# read table into bs4 and get all the tables
bs4page = bs4.BeautifulSoup(page.text, 'html.parser')
tables = bs4page.find_all('table',{'class':"wikitable"})


# for table in tables:
#     print("next")
#     df = pd.read_html(StringIO(str(table)))[0]
#     print(df.head())

gdp_df = pd.read_html(StringIO(str(tables[0])))[0]
print(gdp_df.head())


old_url = "https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467"

# read in the webpage
page_old = rq.get(old_url, headers={"User-Agent": "Chrome"})

# read in old table into bs4 and get all the tables
bs4page = bs4.BeautifulSoup(page_old.text, 'html.parser')
tables_old = bs4page.find_all('table')

# print(len(tables))
# for table in tables:
#     print("next")
#     df = pd.read_html(StringIO(str(table)))[0]
#     print(df.head())

# get table with regions
region_table = pd.read_html(StringIO(str(tables_old[2])))[0]
print(region_table.head())

  Country/Territory IMF (2026)[1] World Bank (2025)[6]  \
0             World     126295331            118350166   
1     United States      32383920             30769700   
2        China[n 1]      20851593             19498039   
3           Germany       5452858              5050923   
4             Japan       4379253              4435163   

  United Nations (2024)[7]  
0                111565144  
1                 29298000  
2                 18743802  
3                  4659929  
4                  4026211  
  Country/Territory UN region IMF[1][13]            World Bank[14]             \
  Country/Territory UN region   Forecast       Year       Estimate       Year   
0             World         —  104476432       2023      100562011       2022   
1     United States  Americas   26949643       2023       25462700       2022   
2             China      Asia   17700899  [n 1]2023       17963171  [n 3]2022   
3           Germany    Europe    4429838       2023        4072192      

In [128]:
# Data Cleaning

# rename IMF column to make it more manageable
gdp_df.rename(columns={'IMF (2026)[1]': 'IMF'}, inplace=True)
print(gdp_df.columns)


# Fill in NAs
gdp_df.loc[gdp_df['IMF'] == "—N/a", 'IMF'] = gdp_df['United Nations (2024)[7]']
gdp_df.loc[gdp_df['IMF'] == "—N/a", 'IMF'] = gdp_df['World Bank (2025)[6]']

# Make sure there are no more NAs
count_nas = (gdp_df['IMF'] == '—N/a').sum()
print("Occurrences of 'NA':", count_nas)


# Flatten the region table column header 
print(region_table.columns)
# avoid unneccesarily joining 
if isinstance(region_table.columns, pd.MultiIndex):
    region_table.columns = ['_'.join(col) for col in region_table.columns]
    region_table = region_table.reset_index()

# change regions table column so it can be joined/merged on
region_table.rename(columns={'Country/Territory_Country/Territory': 'Country/Territory',
                            'UN region_UN region': 'Region'}, inplace=True)

print(region_table.columns)

# Make sure the references numbers don't get in there:
gdp_df = gdp_df.replace(r"\[n \d+\]", "", regex=True)
region_table = region_table.replace(r"\[n \d+\]", "", regex=True)

# print(gdp_df.head())
# print("_______")
# print(region_table.head())

# Merge the data so there is region data with the GDP data
gdp_df = gdp_df.merge(
    region_table,
    on="Country/Territory",
    how="left"
)

# get rid of years in parenthesis
gdp_df["IMF"] = gdp_df["IMF"].astype(str).str.replace(r"\(\d+\)", "", regex=True)

# Change IMF to numbers
gdp_df["IMF"] = gdp_df["IMF"].str.replace(",", "")
gdp_df["IMF"] = gdp_df["IMF"].astype(float)

# Get rid of the world as a country
gdp_df = gdp_df.loc[gdp_df["Region"] != "—"]


print("CLEANED AND MERGED DF")
print(gdp_df.head())

Index(['Country/Territory', 'IMF', 'World Bank (2025)[6]',
       'United Nations (2024)[7]'],
      dtype='str')
Occurrences of 'NA': 0
MultiIndex([( 'Country/Territory', 'Country/Territory'),
            (         'UN region',         'UN region'),
            (        'IMF[1][13]',          'Forecast'),
            (        'IMF[1][13]',              'Year'),
            (    'World Bank[14]',          'Estimate'),
            (    'World Bank[14]',              'Year'),
            ('United Nations[15]',          'Estimate'),
            ('United Nations[15]',              'Year')],
           )
Index(['index', 'Country/Territory', 'Region', 'IMF[1][13]_Forecast',
       'IMF[1][13]_Year', 'World Bank[14]_Estimate', 'World Bank[14]_Year',
       'United Nations[15]_Estimate', 'United Nations[15]_Year'],
      dtype='str')
CLEANED AND MERGED DF
  Country/Territory         IMF World Bank (2025)[6] United Nations (2024)[7]  \
1     United States  32383920.0             30769700       

In [129]:
print(gdp_df.groupby("Region")["IMF"].sum())
print(gdp_df["Region"].value_counts())

Region
Africa       3574118.0
Americas    43309873.0
Asia        44988096.0
Europe      32305975.0
Oceania      2465866.0
Name: IMF, dtype: float64
Region
Africa      55
Asia        51
Americas    46
Europe      44
Oceania     17
Name: count, dtype: int64


In [130]:
# Plot the data
import plotly.express as px

fig = px.bar(
    gdp_df,
    x="Region",
    y="IMF",
    color="Country/Territory",
    title="GDP by Region",
    barmode="stack"
)

fig.show()

In [131]:
# Write the figure to an html file
fig.write_html("stacked_bar.html")

# Part 2

In [133]:
!pip install certifi


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Needed to implement because of SSL Error

import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

print(certifi.where())

/Users/brockmc1/Documents/ds4bme/data_venv/lib/python3.11/site-packages/certifi/cacert.pem


In [164]:
import pandas as pd
import plotly.express as px
import numpy as np
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep = "\t").drop(['Level5'], axis = 1)
multilevel_lookup = multilevel_lookup.rename(columns = {
    "modify"   : "roi",
    "modify.1" : "level4",
    "modify.2" : "level3",
    "modify.3" : "level2",
    "modify.4" : "level1"})
multilevel_lookup = multilevel_lookup[['roi', 'level4', 'level3', 'level2', 'level1']]
multilevel_lookup.head()

,roi,level4,level3,level2,level1
0,SFG_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
1,SFG_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
2,SFG_PFC_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
3,SFG_PFC_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
4,SFG_pole_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L


In [165]:
## Now load in the subject data
id = 127
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == id)]
subjectData = subjectData[['roi', 'volume']]
## Merge the subject data with the multilevel data
subjectData = pd.merge(subjectData, multilevel_lookup, on = "roi")
subjectData = subjectData.assign(icv = "ICV")
subjectData = subjectData.assign(comp = subjectData.volume / np.sum(subjectData.volume))
subjectData.head()

,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


In [166]:
# Make a sunburst plot
fig = px.sunburst(subjectData, path=['icv', 'level1', 'level2', 'level3', 'level4', 'roi'],
                  values='comp', width=800, height=800)
fig.show()

In [ ]:
#ICV -> Level 1, get all the level 1 columns
icv_to_level1 = subjectData.groupby(['icv', 'level1'])["volume"].sum().reset_index()
print(icv_to_level1)

#Level 1 -> Level 2, get all the level 2 columns
level1_to_level2 = subjectData.groupby(['level1', 'level2'])["volume"].sum().reset_index()
print(level1_to_level2)

#Level 2 -> Level 3, get all the level 2 columns
level2_to_level3 = subjectData.groupby(['level2', 'level3'])["volume"].sum().reset_index()
print(level2_to_level3)

# Make a list of all the names within the levels
# Need to distinguish what level they are in (some links were becoming circular)
labels = ["ICV"]
labels.extend(["L1_" + x for x in icv_to_level1["level1"].unique()])
labels.extend(["L2_" + x for x in level1_to_level2["level2"].unique()])
labels.extend(["L3_" + x for x in level2_to_level3["level3"].unique()])

print(labels)


# Arrange into sources, targets, and values for the sankey plot
sources = []
targets = []
values = []

# go through rows of ICV-> level 1 and create links
for index, row in icv_to_level1.iterrows():
    # get index of ICV
    sources.append(labels.index("ICV"))

    # get index of what icv goes to
    targets.append(labels.index("L1_" + row["level1"]))

    # get volume value of that icv -> level1
    values.append(row["volume"])

# go through rows of level 1-> level 2 and create links
for index, row in level1_to_level2.iterrows():
    # get index of level 1 
    sources.append(labels.index("L1_" + row["level1"]))

    # get index of what level1 goes to
    targets.append(labels.index("L2_" + row["level2"]))

    # get volume value of that level1 -> level2
    values.append(row["volume"])

# go through rows of level 2-> level 3 and create links
for index, row in level2_to_level3.iterrows():
    # get index of level 2
    sources.append(labels.index("L2_" + row["level2"]))

    # get index of what level2 goes to
    targets.append(labels.index("L3_" + row["level3"]))

    # get volume value of that level2 -> level3
    values.append(row["volume"])



   icv           level1  volume
0  ICV              CSF  109787
1  ICV   Diencephalon_L   11817
2  ICV   Diencephalon_R   11560
3  ICV    Mesencephalon   10271
4  ICV    Metencephalon  159411
5  ICV   Myelencephalon    4975
6  ICV  Telencephalon_L  531153
7  ICV  Telencephalon_R  543444
             level1            level2  volume
0               CSF          Sulcus_L   33975
1               CSF          Sulcus_R   30018
2               CSF         Ventricle   44979
3               CSF        Ventricle      815
4    Diencephalon_L  BasalForebrain_L    5475
5    Diencephalon_L        Thalamus_L    6342
6    Diencephalon_R  BasalForebrain_R    5188
7    Diencephalon_R        Thalamus_R    6372
8     Mesencephalon   Mesencephalon_L    4945
9     Mesencephalon   Mesencephalon_R    5326
10    Metencephalon   Metencephalon_L   79498
11    Metencephalon   Metencephalon_R   79913
12   Myelencephalon  Myelencephalon_L    2403
13   Myelencephalon  Myelencephalon_R    2572
14  Telencephalon_L  C

In [168]:
# Plot Sankey graph 

import plotly.graph_objects as go

# Remove level prefixes
display_labels = []
for label in labels:
    # Keep ICV as it is
    if label == "ICV":
        display_labels.append(label)
    else:
        # get rid of the first 3 characters of every other label
        display_labels.append(label[3:])

fig = go.Figure(data=[go.Sankey(
    node = dict(
      label = display_labels,
    ),
    link = dict(
      source = sources,
      target = targets,
      value = values
  ))])

fig.update_layout(title_text="MRICloud Sankey Diagram", font_size=10)
fig.show()

In [169]:
# Write figure to html
fig.write_html("sankey.html")

[View Sankey Website](https://mcbrock.github.io/DS4BME_Fall_2026/)